In [1]:
from pyspark.sql import SparkSession
from datetime import datetime


In [2]:
spark = SparkSession.builder.appName("myAPP").getOrCreate()

# Raw Data

In [3]:

from pyspark.sql.types import StructType, StringType, StructField

raw_schema = StructType([
    StructField("order_id",    StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("city",        StringType(), True),
    StructField("category",    StringType(), True),
    StructField("product",     StringType(), True),
    StructField("amount",      StringType(), True),
    StructField("order_date",  StringType(), True),
    StructField("status",      StringType(), True),
])


In [5]:
RAW_PATH = "/content/orders_raw.csv"

In [6]:

orders_raw_df = (spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(raw_schema)
    .csv(RAW_PATH)
)


In [7]:

orders_raw_df.printSchema()
print("Total records:", orders_raw_df.count())


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)

Total records: 300000


In [8]:

import pandas as pd

RAW_PATH = "/content/orders_raw.csv"
CLEAN_PATH = "/content/orders_cleaned.csv"

df = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False).fillna("")

df["order_date_parsed"] = pd.to_datetime(
    df["order_date"],
    errors="coerce",
    infer_datetime_format=True
)

df["order_date_iso"] = df["order_date_parsed"].dt.strftime("%Y-%m-%d")
df["order_date_iso"] = df["order_date_iso"].fillna("")

df["order_year"]  = df["order_date_parsed"].dt.year.astype("Int64").astype(str).replace("<NA>", "")
df["order_month"] = df["order_date_parsed"].dt.month.astype("Int64").astype(str).replace("<NA>", "")
df["order_day"]   = df["order_date_parsed"].dt.day.astype("Int64").astype(str).replace("<NA>", "")

df.to_csv(CLEAN_PATH, index=False)
print("Wrote cleaned CSV to:", CLEAN_PATH)
print(df.head(3))


/tmp/ipython-input-4104799838.py:11: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["order_date_parsed"] = pd.to_datetime(


Wrote cleaned CSV to: /content/orders_cleaned.csv
      order_id customer_id         city     category product   amount  \
0  ORD00000000     C000000   hyderabad      grocery     Oil   invalid   
1  ORD00000001     C000001         Pune      Grocery   Sugar    35430   
2  ORD00000002     C000002         Pune  Electronics  Mobile    65358   

   order_date     status order_date_parsed order_date_iso order_year  \
0  01/01/2024  Cancelled        2024-01-01     2024-01-01       2024   
1  2024-01-02  Completed               NaT                             
2  2024-01-03  Completed               NaT                             

  order_month order_day  
0           1         1  
1                        
2                        


In [10]:

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, DateType

CLEAN_PATH = "/content/orders_cleaned.csv"

clean_schema = StructType([
    StructField("order_id",       StringType(), True),
    StructField("customer_id",    StringType(), True),
    StructField("city",           StringType(), True),
    StructField("category",       StringType(), True),
    StructField("product",        StringType(), True),
    StructField("amount",         StringType(), True),
    StructField("order_date",     StringType(), True),
    StructField("status",         StringType(), True),
    StructField("order_date_parsed", StringType(), True),
    StructField("order_date_iso", StringType(), True),
    StructField("order_year",     StringType(), True),
    StructField("order_month",    StringType(), True),
    StructField("order_day",      StringType(), True),
])

orders_clean_df = (spark.read
    .option("header", True)
    .option("inferSchema", False)
    .schema(clean_schema)
    .csv(CLEAN_PATH)
)


from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, DateType

amount_sanitized = F.regexp_replace(F.trim(F.col("amount")), r"[^0-9\.\-]", "")
amount_nullified = F.when(F.length(amount_sanitized) == 0, F.lit(None)).otherwise(amount_sanitized)

orders_typed_df = (orders_clean_df
    .withColumn("amount_clean", amount_nullified)
    .withColumn("amount_num", F.expr("try_cast(amount_clean as DECIMAL(18,2))"))
    .withColumn("order_date_dt", F.to_date(F.col("order_date_iso"), "yyyy-MM-dd"))
    .withColumn("order_year_int",  F.col("order_year").cast("int"))
    .withColumn("order_month_int", F.col("order_month").cast("int"))
    .withColumn("order_day_int",   F.col("order_day").cast("int"))
    .drop("amount_clean")
)

orders_typed_df.printSchema()
print("Total records (typed):", orders_typed_df.count())

orders_typed_df.select(
    "order_id", "amount", "amount_num", "order_date", "order_date_iso", "order_date_dt",
    "order_year_int", "order_month_int", "order_day_int"
).show(20, truncate=False)



root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date_parsed: string (nullable = true)
 |-- order_date_iso: string (nullable = true)
 |-- order_year: string (nullable = true)
 |-- order_month: string (nullable = true)
 |-- order_day: string (nullable = true)
 |-- amount_num: decimal(18,2) (nullable = true)
 |-- order_date_dt: date (nullable = true)
 |-- order_year_int: integer (nullable = true)
 |-- order_month_int: integer (nullable = true)
 |-- order_day_int: integer (nullable = true)

Total records (typed): 300000
+-----------+-------+----------+----------+--------------+-------------+--------------+---------------+-------------+
|order_id   |amount |amount_num|order_date|order_date_iso|order_date_dt|orde

#  Clean leading/trailing spaces from string columns.

In [11]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

src_df = orders_typed_df

string_cols = [f.name for f in src_df.schema.fields if isinstance(f.dataType, StringType)]

def normalize_string_col(col_name):
    col = F.col(col_name)
    return (
        F.when(F.length(F.trim(col)) == 0, F.lit(None))
         .otherwise(F.regexp_replace(F.trim(col), r"\s+", " "))
    )

normalized_df = src_df
for c in string_cols:
    normalized_df = normalized_df.withColumn(c, normalize_string_col(c))

normalized_df.select(string_cols).show(5, truncate=False)


+-----------+-----------+---------+-----------+-----------+-------+----------+---------+-----------------+--------------+----------+-----------+---------+
|order_id   |customer_id|city     |category   |product    |amount |order_date|status   |order_date_parsed|order_date_iso|order_year|order_month|order_day|
+-----------+-----------+---------+-----------+-----------+-------+----------+---------+-----------------+--------------+----------+-----------+---------+
|ORD00000000|C000000    |hyderabad|grocery    |Oil        |invalid|01/01/2024|Cancelled|2024-01-01       |2024-01-01    |2024      |1          |1        |
|ORD00000001|C000001    |Pune     |Grocery    |Sugar      |35430  |2024-01-02|Completed|NULL             |NULL          |NULL      |NULL       |NULL     |
|ORD00000002|C000002    |Pune     |Electronics|Mobile     |65358  |2024-01-03|Completed|NULL             |NULL          |NULL      |NULL       |NULL     |
|ORD00000003|C000003    |Bangalore|Electronics|Laptop     |5558   |202

#Standardize city , category , and product values

In [12]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

src_df = orders_typed_df


string_cols = [f.name for f in src_df.schema.fields if isinstance(f.dataType, StringType)]

def normalize_string_col(col_name):
    col = F.col(col_name)
    return (
        F.when(F.length(F.trim(col)) == 0, F.lit(None))
         .otherwise(F.regexp_replace(F.trim(col), r"\s+", " "))
    )

normalized_df = src_df
for c in string_cols:
    normalized_df = normalized_df.withColumn(c, normalize_string_col(c))

normalized_df.select(string_cols).show(5, truncate=False)


+-----------+-----------+---------+-----------+-----------+-------+----------+---------+-----------------+--------------+----------+-----------+---------+
|order_id   |customer_id|city     |category   |product    |amount |order_date|status   |order_date_parsed|order_date_iso|order_year|order_month|order_day|
+-----------+-----------+---------+-----------+-----------+-------+----------+---------+-----------------+--------------+----------+-----------+---------+
|ORD00000000|C000000    |hyderabad|grocery    |Oil        |invalid|01/01/2024|Cancelled|2024-01-01       |2024-01-01    |2024      |1          |1        |
|ORD00000001|C000001    |Pune     |Grocery    |Sugar      |35430  |2024-01-02|Completed|NULL             |NULL          |NULL      |NULL       |NULL     |
|ORD00000002|C000002    |Pune     |Electronics|Mobile     |65358  |2024-01-03|Completed|NULL             |NULL          |NULL      |NULL       |NULL     |
|ORD00000003|C000003    |Bangalore|Electronics|Laptop     |5558   |202

# Standardize city, category, and product

In [13]:

def standardize_text(col):
    return F.initcap(
        F.regexp_replace(
            F.lower(F.regexp_replace(col, r"[_\-]+", " ")),
            r"\s+",
            " "
        )
    )

standardized_df = (normalized_df
    .withColumn("city_std",     standardize_text(F.col("city")))
    .withColumn("category_std", standardize_text(F.col("category")))
    .withColumn("product_std",  standardize_text(F.col("product")))
)

standardized_df.select("city", "city_std", "category", "category_std", "product", "product_std").show(10, truncate=False)


+---------+---------+-----------+------------+-----------+-----------+
|city     |city_std |category   |category_std|product    |product_std|
+---------+---------+-----------+------------+-----------+-----------+
|hyderabad|Hyderabad|grocery    |Grocery     |Oil        |Oil        |
|Pune     |Pune     |Grocery    |Grocery     |Sugar      |Sugar      |
|Pune     |Pune     |Electronics|Electronics |Mobile     |Mobile     |
|Bangalore|Bangalore|Electronics|Electronics |Laptop     |Laptop     |
|Pune     |Pune     |Home       |Home        |AirPurifier|Airpurifier|
|Delhi    |Delhi    |Fashion    |Fashion     |Jeans      |Jeans      |
|Delhi    |Delhi    |Grocery    |Grocery     |Sugar      |Sugar      |
|Pune     |Pune     |Grocery    |Grocery     |Rice       |Rice       |
|Bangalore|Bangalore|Fashion    |Fashion     |Jeans      |Jeans      |
|Kolkata  |Kolkata  |Electronics|Electronics |Laptop     |Laptop     |
+---------+---------+-----------+------------+-----------+-----------+
only s

# Convert amount to integer safely

In [14]:

from pyspark.sql.types import IntegerType

amount_sanitized = F.regexp_replace(F.trim(F.col("amount")), r"[^0-9\.\-]", "")
amount_clean = F.when(F.length(amount_sanitized) == 0, F.lit(None)).otherwise(amount_sanitized)

has_amount_num = "amount_num" in standardized_df.columns

candidate_double = (F.col("amount_num").cast("double") if has_amount_num else F.lit(None))
candidate_double = F.coalesce(candidate_double, F.expr("try_cast(amount_clean as DOUBLE)"))

typed_df = (standardized_df
    .withColumn("amount_clean", amount_clean)
    .withColumn("amount_candidate", candidate_double)
    .withColumn("amount_int", F.round(F.col("amount_candidate")).cast("int"))
    .withColumn("amount_int", F.when(F.col("amount_int") < 0, F.lit(None)).otherwise(F.col("amount_int")))
    .drop("amount_candidate", "amount_clean")
)

typed_df.select("amount", *(["amount_num"] if has_amount_num else []), "amount_int").show(20, truncate=False)

+-------+----------+----------+
|amount |amount_num|amount_int|
+-------+----------+----------+
|invalid|NULL      |NULL      |
|35430  |35430.00  |35430     |
|65358  |65358.00  |65358     |
|5558   |5558.00   |5558      |
|33659  |33659.00  |33659     |
|8521   |8521.00   |8521      |
|42383  |42383.00  |42383     |
|45362  |45362.00  |45362     |
|10563  |10563.00  |10563     |
|63715  |63715.00  |63715     |
|66576  |66576.00  |66576     |
|50318  |50318.00  |50318     |
|84768  |84768.00  |84768     |
|79121  |79121.00  |79121     |
|79469  |79469.00  |79469     |
|81018  |81018.00  |81018     |
|64225  |64225.00  |64225     |
|69582  |69582.00  |69582     |
|50424  |50424.00  |50424     |
|invalid|NULL      |NULL      |
+-------+----------+----------+
only showing top 20 rows


# Parse order_date supporting multiple formats

In [25]:

from pyspark.sql import functions as F

df = orders_typed_df

spark.conf.set("spark.sql.session.timeZone", "UTC")

date_formats = [
    "yyyy-MM-dd",
    "dd-MM-yyyy",
    "MM-dd-yyyy",
    "dd/MM/yyyy",
    "MM/dd/yyyy",
    "yyyy/MM/dd",
    "dd-MMM-yyyy",
    "MMM dd, yyyy",
    "yyyy-MM-dd HH:mm:ss",
    "dd/MM/yyyy HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss.SSS",
    "yyyy-MM-dd'T'HH:mm:ssXXX",
    "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"
]

def try_ts_expr(col_name: str, fmt: str):
    fmt_sql = fmt.replace("'", "''")
    return F.expr(f"try_to_timestamp({col_name}, '{fmt_sql}')")

ts_candidates = []

if "order_date_iso" in df.columns:
    ts_candidates.append(try_ts_expr("order_date_iso", "yyyy-MM-dd"))

for fmt in date_formats:
    ts_candidates.append(try_ts_expr("order_date", fmt))

ts_candidates.append(F.expr("try_to_timestamp(order_date)"))

order_ts = F.coalesce(*ts_candidates)



In [20]:

parsed_df = (df
    .withColumn("order_ts", order_ts)
    .withColumn("order_date_dt", F.to_date(F.col("order_ts")))
    .withColumn("order_year_int",  F.year(F.col("order_ts")))
    .withColumn("order_month_int", F.month(F.col("order_ts")))
    .withColumn("order_day_int",   F.dayofmonth(F.col("order_ts"))))


In [26]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

string_cols = [f.name for f in parsed_df.schema.fields if isinstance(f.dataType, StringType)]

def normalize_string_col(col_name):
    col = F.col(col_name)
    return (
        F.when(F.length(F.trim(col)) == 0, F.lit(None))
         .otherwise(F.regexp_replace(F.trim(col), r"\s+", " "))
    )

normalized_df = parsed_df
for c in string_cols:
    normalized_df = normalized_df.withColumn(c, normalize_string_col(c))


In [27]:

def standardize_text(col):
    return F.initcap(
        F.regexp_replace(
            F.lower(F.regexp_replace(col, r"[_\-]+", " ")),
            r"\s+",
            " "
        )
    )


In [28]:

standardized_df = (normalized_df
    .withColumn("city_std",     standardize_text(F.col("city")))
    .withColumn("category_std", standardize_text(F.col("category")))
    .withColumn("product_std",  standardize_text(F.col("product")))
)


In [29]:

amount_sanitized = F.regexp_replace(F.trim(F.col("amount")), r"[^0-9\.\-]", "")
amount_clean = F.when(F.length(amount_sanitized) == 0, F.lit(None)).otherwise(amount_sanitized)

has_amount_num = "amount_num" in standardized_df.columns

candidate_double = (F.col("amount_num").cast("double") if has_amount_num else F.lit(None))
candidate_double = F.coalesce(candidate_double, F.expr("try_cast(amount as DOUBLE)"), F.expr("try_cast(amount_clean as DOUBLE)"))

typed_df = (standardized_df
    .withColumn("amount_clean", amount_clean)
    .withColumn("amount_candidate", candidate_double)
    .withColumn("amount_int", F.expr("try_cast(round(amount_candidate) as INT)"))
    .withColumn("amount_int", F.when(F.col("amount_int") < 0, F.lit(None)).otherwise(F.col("amount_int")))
    .drop("amount_candidate", "amount_clean")
)


# Identify & handle invalid or null records (build DQ flags and reasons)

In [30]:

dq_df = (typed_df
    .withColumn("dq_missing_order_id",  F.col("order_id").isNull())
    .withColumn("dq_missing_customer",  F.col("customer_id").isNull())
    .withColumn("dq_bad_amount",        F.col("amount_int").isNull())
    .withColumn("dq_bad_order_date",    F.col("order_date_dt").isNull())
    .withColumn(
        "dq_reason",
        F.array_remove(F.array(
            F.when(F.col("dq_missing_order_id"),  F.lit("missing_order_id")),
            F.when(F.col("dq_missing_customer"),  F.lit("missing_customer_id")),
            F.when(F.col("dq_bad_amount"),        F.lit("invalid_amount")),
            F.when(F.col("dq_bad_order_date"),    F.lit("invalid_order_date"))
        ), F.lit(None))
    )
    .withColumn("dq_reason_str",
        F.when(F.size(F.col("dq_reason")) == 0, F.lit(None))
         .otherwise(F.concat_ws(",", F.col("dq_reason")))
    )
    .withColumn("is_valid_row",
        ~(F.col("dq_missing_order_id") | F.col("dq_missing_customer")
        | F.col("dq_bad_amount") | F.col("dq_bad_order_date")))
)

valid_df   = dq_df.filter(F.col("is_valid_row"))
invalid_df = dq_df.filter(~F.col("is_valid_row"))


# Clean leading/trailing spaces from string columns

In [33]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

df = orders_typed_df

string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

def normalize_string_col(col_name):
    col = F.col(col_name)
    return (
        F.when(F.length(F.trim(col)) == 0, F.lit(None))
         .otherwise(F.regexp_replace(F.trim(col), r"\s+", " "))
    )

clean_df = df
for c in string_cols:
    clean_df = clean_df.withColumn(c, normalize_string_col(c))

clean_df.select(string_cols).show(5, truncate=False)


+-----------+-----------+---------+-----------+-----------+-------+----------+---------+-----------------+--------------+----------+-----------+---------+
|order_id   |customer_id|city     |category   |product    |amount |order_date|status   |order_date_parsed|order_date_iso|order_year|order_month|order_day|
+-----------+-----------+---------+-----------+-----------+-------+----------+---------+-----------------+--------------+----------+-----------+---------+
|ORD00000000|C000000    |hyderabad|grocery    |Oil        |invalid|01/01/2024|Cancelled|2024-01-01       |2024-01-01    |2024      |1          |1        |
|ORD00000001|C000001    |Pune     |Grocery    |Sugar      |35430  |2024-01-02|Completed|NULL             |NULL          |NULL      |NULL       |NULL     |
|ORD00000002|C000002    |Pune     |Electronics|Mobile     |65358  |2024-01-03|Completed|NULL             |NULL          |NULL      |NULL       |NULL     |
|ORD00000003|C000003    |Bangalore|Electronics|Laptop     |5558   |202

#  Remove duplicate records based on order_id .
# Filter only records with status = Completed .
# Validate record counts before and after ltering

In [36]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = clean_df

total_before = df.count()
print(f"[Baseline] Total rows before de-dup/filter: {total_before}")


w = Window.partitionBy(F.trim(F.col("order_id"))).orderBy(
    F.col("order_ts").asc_nulls_last() if "order_ts" in df.columns else F.lit(1)
)

dedup_df = (df
    .withColumn("order_id_trim", F.trim(F.col("order_id")))
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

count_after_dedup = dedup_df.count()
dupe_removed = total_before - count_after_dedup
print(f"[De-dup] Rows after de-dup by order_id: {count_after_dedup} | Duplicates removed: {dupe_removed}")

completed_df = dedup_df.filter(
    F.lower(F.trim(F.col("status"))) == F.lit("completed")
)

count_after_filter = completed_df.count()
filtered_out = count_after_dedup - count_after_filter
print(f"[Filter] Rows with status='Completed': {count_after_filter} | Rows filtered out: {filtered_out}")

print("=== Validation Summary ===")
print(f"Initial rows                : {total_before}")
print(f"After de-dup by order_id    : {count_after_dedup} (removed {dupe_removed})")
print(f"After status='Completed'    : {count_after_filter} (filtered {filtered_out})")


completed_df.select("order_id", "customer_id", "status").show(10, truncate=False)


[Baseline] Total rows before de-dup/filter: 300000
[De-dup] Rows after de-dup by order_id: 300000 | Duplicates removed: 0
[Filter] Rows with status='Completed': 285000 | Rows filtered out: 15000
=== Validation Summary ===
Initial rows                : 300000
After de-dup by order_id    : 300000 (removed 0)
After status='Completed'    : 285000 (filtered 15000)
+-----------+-----------+---------+
|order_id   |customer_id|status   |
+-----------+-----------+---------+
|ORD00000001|C000001    |Completed|
|ORD00000007|C000007    |Completed|
|ORD00000008|C000008    |Completed|
|ORD00000010|C000010    |Completed|
|ORD00000011|C000011    |Completed|
|ORD00000012|C000012    |Completed|
|ORD00000014|C000014    |Completed|
|ORD00000015|C000015    |Completed|
|ORD00000017|C000017    |Completed|
|ORD00000019|C000019    |Completed|
+-----------+-----------+---------+
only showing top 10 rows


#  Identify operations that cause shu es.
# Use explain(True) to analyze the execution plan.
# Apply repartitioning to optimize aggregations.

In [37]:

df_plan = completed_df

agg_df = (df_plan
          .groupBy("city_std", "category_std")
          .agg(
              F.count("*").alias("order_cnt"),
              F.sum("amount_int").alias("total_amount_int")
          ))

print("=== Execution Plan (before repartition) ===")
agg_df.explain(True)


{"ts": "2026-01-05 11:38:29.629", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `city_std` cannot be resolved. Did you mean one of the following? [`city`, `amount`, `status`, `category`, `product`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor26.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1838.agg.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `city_std` cannot be resolved. Did you mean one of the following? [`city`, `amount`, `status`, `category`, `product`]. SQLSTATE: 42703;\n'Aggregate ['city_std, 'category_std], ['city_std, 'category_std, count(1) AS order_cnt#783L, 'sum('amount_int) AS total_amount_int#784]\n+

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `city_std` cannot be resolved. Did you mean one of the following? [`city`, `amount`, `status`, `category`, `product`]. SQLSTATE: 42703;
'Aggregate ['city_std, 'category_std], ['city_std, 'category_std, count(1) AS order_cnt#783L, 'sum('amount_int) AS total_amount_int#784]
+- Filter (lower(trim(status#548, None)) = completed)
   +- Project [order_id#541, customer_id#542, city#543, category#544, product#545, amount#546, order_date#547, status#548, order_date_parsed#549, order_date_iso#550, order_year#551, order_month#552, order_day#553, amount_num#131, order_date_dt#132, order_year_int#133, order_month_int#134, order_day_int#135, order_id_trim#717]
      +- Filter (rn#718 = 1)
         +- Project [order_id#541, customer_id#542, city#543, category#544, product#545, amount#546, order_date#547, status#548, order_date_parsed#549, order_date_iso#550, order_year#551, order_month#552, orde

# Rank cities by total revenue.
# Rank products within each category by revenue.
# Identify the top product per category

In [41]:


INPUT = "/content/orders_cleaned.csv"
df = (spark.read
      .option("header", True)
      .option("inferSchema", False)
      .csv(INPUT))


def normalize_string_col(col):
    return F.when(F.length(F.trim(col)) == 0, F.lit(None)) \
            .otherwise(F.regexp_replace(F.trim(col), r"\s+", " "))

def standardize_text(col):
    return F.initcap(F.regexp_replace(F.lower(F.regexp_replace(col, r"[_\-]+", " ")), r"\s+", " "))

df = (df
      .withColumn("city",     normalize_string_col(F.col("city")))
      .withColumn("category", normalize_string_col(F.col("category")))
      .withColumn("product",  normalize_string_col(F.col("product")))
      .withColumn("city_std",     standardize_text(F.col("city")))
      .withColumn("category_std", standardize_text(F.col("category")))
      .withColumn("product_std",  standardize_text(F.col("product")))
)


amount_sanitized = F.regexp_replace(F.trim(F.col("amount")), r"[^0-9\.\-]", "")
amount_clean = F.when(F.length(amount_sanitized) == 0, F.lit(None)).otherwise(amount_sanitized)

df = (df
      .withColumn("amount_clean", amount_clean)
      .withColumn("amount_double", F.expr("try_cast(amount_clean as DOUBLE)"))
      .withColumn("amount_int", F.expr("try_cast(round(amount_double) as INT)"))
      .drop("amount_double", "amount_clean")
)


df_fact = df.filter(F.col("amount_int").isNotNull())

city_rev = (df_fact
    .groupBy("city_std")
    .agg(F.sum("amount_int").alias("total_revenue"))
)

w_city = Window.orderBy(F.desc("total_revenue"))
city_ranked = (city_rev
    .withColumn("city_rank", F.dense_rank().over(w_city))
    .orderBy(F.col("city_rank").asc(), F.col("city_std").asc())
)

print("=== Rank Cities by Total Revenue ===")
city_ranked.show(50, truncate=False)

prod_rev_by_cat = (df_fact
    .groupBy("category_std", "product_std")
    .agg(F.sum("amount_int").alias("total_revenue"))
)

w_cat_prod = Window.partitionBy("category_std").orderBy(F.desc("total_revenue"), F.col("product_std").asc())
prod_ranked_within_cat = (prod_rev_by_cat
    .withColumn("product_rank_in_category", F.dense_rank().over(w_cat_prod))
    .orderBy(F.col("category_std").asc(), F.col("product_rank_in_category").asc())
)

print("=== Rank Products within Each Category by Revenue ===")
prod_ranked_within_cat.show(50, truncate=False)

top_product_per_category = prod_ranked_within_cat.filter(F.col("product_rank_in_category") == 1)

print("=== Top Product per Category ===")
top_product_per_category.show(100, truncate=False)


=== Rank Cities by Total Revenue ===
+---------+-------------+---------+
|city_std |total_revenue|city_rank|
+---------+-------------+---------+
|Pune     |1733418439   |1        |
|Hyderabad|1731794799   |2        |
|Delhi    |1719458186   |3        |
|Chennai  |1714214871   |4        |
|Bangalore|1713862477   |5        |
|Mumbai   |1712702171   |6        |
|Kolkata  |1708896076   |7        |
+---------+-------------+---------+

=== Rank Products within Each Category by Revenue ===
+------------+-----------+-------------+------------------------+
|category_std|product_std|total_revenue|product_rank_in_category|
+------------+-----------+-------------+------------------------+
|Electronics |Laptop     |1014441137   |1                       |
|Electronics |Tablet     |1011526294   |2                       |
|Electronics |Mobile     |993672519    |3                       |
|Fashion     |Jeans      |1001087409   |1                       |
|Fashion     |Shoes      |994655138    |2         

# Write the cleaned dataset to Parquet partitioned by city

In [42]:

OUTPUT_PARQUET = "/content/orders_parquet_by_city"

clean_to_write = completed_df.select(
    "order_id", "customer_id",
    "city_std", "category_std", "product_std",
    "amount_int", "order_date_dt", "order_year_int", "order_month_int", "order_day_int",
    "status"
)

(clean_to_write
 .write
 .mode("overwrite")
 .partitionBy("city_std")
 .parquet(OUTPUT_PARQUET))


{"ts": "2026-01-05 11:47:16.539", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `city_std` cannot be resolved. Did you mean one of the following? [`city`, `amount`, `status`, `category`, `product`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor26.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1826.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `city_std` cannot be resolved. Did you mean one of the following? [`city`, `amount`, `status`, `category`, `product`]. SQLSTATE: 42703;\n'Project [order_id#541, customer_id#542, 'city_std, 'category_std, 'product_std, 'amount_int, order_date_dt#132, order_year_int#133, ord

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `city_std` cannot be resolved. Did you mean one of the following? [`city`, `amount`, `status`, `category`, `product`]. SQLSTATE: 42703;
'Project [order_id#541, customer_id#542, 'city_std, 'category_std, 'product_std, 'amount_int, order_date_dt#132, order_year_int#133, order_month_int#134, order_day_int#135, status#548]
+- Filter (lower(trim(status#548, None)) = completed)
   +- Project [order_id#541, customer_id#542, city#543, category#544, product#545, amount#546, order_date#547, status#548, order_date_parsed#549, order_date_iso#550, order_year#551, order_month#552, order_day#553, amount_num#131, order_date_dt#132, order_year_int#133, order_month_int#134, order_day_int#135, order_id_trim#717]
      +- Filter (rn#718 = 1)
         +- Project [order_id#541, customer_id#542, city#543, category#544, product#545, amount#546, order_date#547, status#548, order_date_parsed#549, order_dat

# Why CSV is not suitable for analytics output

Row-oriented, no schema, No compression/indexing, no nested/complex types, error-prone parsing

# Why does this line cause failure?

df = df.filter(df.amount > 50000).show()

This causes a failure because .show() is assiging null type to df.